# Análise de umidade do leito

Lê um `gpslogNN.csv` gerado pelo firmware e plota a série de umidade e os pontos georreferenciados.

Instale as dependências com `pip install -r examples/requirements.txt`.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

LOG = Path("../firmware/moni_humid_leitos_rios/gpslog00.csv")
df = pd.read_csv(LOG)
df.head()

In [ ]:
# Integridade: valida o CRC-8 de cada linha.
def crc8(text: str) -> int:
    crc = 0
    for ch in text.encode():
        crc ^= ch
        for _ in range(8):
            crc = ((crc << 1) ^ 0x07) & 0xFF if crc & 0x80 else (crc << 1) & 0xFF
    return crc

rows = LOG.read_text().splitlines()
valid = []
for line in rows[1:]:
    body, _, check = line.rpartition(",")
    valid.append(int(check, 16) == crc8(body))
print(f"Linhas íntegras: {sum(valid)}/{len(valid)}")

In [ ]:
df["timestamp"] = pd.to_datetime(
    df["date"].astype(str) + " " + df["time"].astype(str),
    format="%d/%m/%Y %H:%M:%S", errors="coerce",
)

ax = df.plot(x="timestamp", y="moisture_pct", marker="o", legend=False)
ax.set(xlabel="Data/hora (UTC)", ylabel="Umidade (%)", title="Série de umidade")
plt.tight_layout()

In [ ]:
geo = df[df["gps_valid"] == 1]
ax = geo.plot.scatter(x="longitude", y="latitude", c="moisture_pct",
                      cmap="viridis", colorbar=True, s=40)
ax.set(xlabel="Longitude", ylabel="Latitude", title="Pontos de medição")
plt.tight_layout()